In [ ]:
"""INSERIMENTO OPENAI_API_KEY HIDED"""

from getpass import getpass
import os
import openai

api_key = getpass("Inserisci la tua OPENAI_API_KEY: ").strip()

if not api_key:
    raise ValueError("Chiave OPENAI_API_KEY non può essere vuota. Inseriscila correttamente.")

os.environ["OPENAI_API_KEY"] = api_key
openai.api_key = api_key

print("Chiave API impostata correttamente. Non mostrare la chiave in output per sicurezza.")

Inserisci la tua OPENAI_API_KEY: ··········
Chiave API impostata correttamente. Non mostrare la chiave in output per sicurezza.


In [ ]:
"""SETUP API KEY: Imposto la variabile d'ambiente OPEN_API_KEY  e ne verifivo la dsponibilità"""

import openai
openai.api_key = os.getenv("OPENAI_API_KEY")
print("OPENAI_API_KEY impostata?", bool(os.getenv("OPENAI_API_KEY")))


OPENAI_API_KEY impostata? True


In [ ]:
"""INPUT: Carico gli input forniti (trascrizione del meeting dal link)"""

import requests

MEETING_TRANSCRIPTION_URL = (
    "https://raw.githubusercontent.com/Profession-AI/progetti-llm/refs/heads/main/"
    "Riassunto%20e%20identificazione%20delle%20attivit%C3%A0%20dalla%20trascrizione%20di%20un%20meeting/"
    "meeting_transcription.txt"
)


def load_meeting_transcription(url: str) -> str:
    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        text = response.text.strip()
        if not text:
            raise RuntimeError("La trascrizione scaricata è vuota.")
        return text
    except requests.RequestException as e:
        print(f"Errore durante il download della trascrizione: {e}")
        return ""



In [ ]:
"""PRE PROCESSING: normalizzo il testo e preparo una di lista di turni di parola"""

import re
from dataclasses import dataclass
from typing import List
import string
import nltk
from nltk.corpus import stopwords
import nltk
nltk.download("stopwords")
# Scarica stopwords una volta (decommenta se non fatto)
# nltk.download('stopwords')

stop_words = set(stopwords.words("italian"))

@dataclass
class PreprocessResult:
    transcript_normalized: str
    turns: List[str]


def normalize_text(text: str) -> str:
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def insert_newlines_before_speakers(text: str) -> str:
    # Regex per nomi speaker in formato NOME COGNOME: case insensitive,
    # nomi con lettere accentate e apostrofi inclusi
    speaker_pat = r"([A-ZÀ-ÖØ-Ý][\w’'`-]+ [A-ZÀ-ÖØ-Ý][\w’'`-]+:)"
    return re.sub(rf"(?<!\n){speaker_pat}", r"\n\1", text, flags=re.IGNORECASE)


def split_turns(text: str) -> List[str]:
    return [line.strip() for line in text.split("\n") if line.strip()]


def clean_text(text: str) -> str:
    # Rimuove tag tipo [rumore], [ah], ecc.
    text = re.sub(r"$\w+$", "", text)
    # Minuscole
    text = text.lower()
    # Rimuove punteggiatura
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Rimuove stopwords
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]
    # Ricompone in stringa
    return " ".join(tokens)


def preprocess_transcript(transcript: str) -> PreprocessResult:
    t = normalize_text(transcript)
    t = insert_newlines_before_speakers(t)
    turns = split_turns(t)

    # Pulisci ciascun turno
    cleaned_turns = [clean_text(turn) for turn in turns]

    return PreprocessResult(transcript_normalized=t, turns=cleaned_turns)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
""""RIASSUNTO LLM CON API KEY: genero un riassunto in bullet points usando LLM"""

from typing import List


def has_openai_key() -> bool:
    import os
    return bool(os.getenv("OPENAI_API_KEY"))


def openai_text(prompt: str, model: str = "gpt-4.1-mini") -> str:
    from openai import OpenAI  # pip install openai

    client = OpenAI()
    resp = client.responses.create(
        model=model,
        input=prompt,
        temperature=0.2,
    )

    txt = getattr(resp, "output_text", None)
    if isinstance(txt, str) and txt.strip():
        return txt.strip()

    parts = []
    for item in resp.output:
        if getattr(item, "type", None) == "message":
            for c in item.content:
                if getattr(c, "type", None) in ("output_text", "text"):
                    parts.append(getattr(c, "text", "") or "")
    return "\n".join(p for p in parts if p).strip()


def offline_summary(transcript: str, max_points: int = 8) -> List[str]:
    keywords = [
        "interfaccia", "ui", "fatture", "pagamenti", "scadenze", "notifiche",
        "promemoria", "smtp", "email", "report", "dashboard", "filtri",
        "grafici", "cloud", "archiviazione", "pdf", "ricerca", "piano", "modulo"
    ]
    points = []
    for line in split_turns(transcript):
        ll = line.lower()
        if any(k in ll for k in keywords):
            cleaned = re.sub(r"^[A-ZÀ-ÖØ-Ý][\w’'-]+(?: [A-ZÀ-ÖØ-Ý][\w’'-]+)+:\s*", "", line).strip()
            if cleaned and cleaned not in points:
                points.append(cleaned)
    return points[:max_points] if points else ["(Fallback offline: nessun punto chiave rilevato)"]


def llm_summary(transcript_normalized: str, model: str = "gpt-4.1-mini") -> List[str]:
    prompt = f"""
Genera un riassunto CONCISO della seguente trascrizione.
Vincoli:
- 5-8 bullet points
- massimo 12 parole per bullet
- non inventare nulla (usa solo informazioni nel testo)
Output: SOLO bullet, una riga per bullet, prefisso "- ".

TRASCRIZIONE:
\"\"\"{transcript_normalized}\"\"\"
"""
    raw = openai_text(prompt, model=model)

    bullets = []
    for line in raw.splitlines():
        line = line.strip()
        if not line:
            continue
        if line.startswith("-"):
            bullets.append(line[1:].strip())
    return bullets or ["(LLM: output non in formato bullet)"]


def generate_summary(transcript_normalized: str, model: str = "gpt-4.1-mini") -> List[str]:
    if has_openai_key():
        return llm_summary(transcript_normalized, model=model)
    return offline_summary(transcript_normalized)

In [ ]:
"""ESTRAZIONE ATTIVITA' CON LLM: Estraggo le attività assegnate usando LLM via OpenAI e restituisco una lista di task e a chi sono assegnate"""

import json
from dataclasses import dataclass
from typing import List


@dataclass
class Task:
    assignee: str
    task: str
    evidence: str


def require_openai_key() -> None:
    import os
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("OPENAI_API_KEY non trovata: impostala nel BLOCCO 0.")


def llm_extract_tasks(transcript_normalized: str, model: str = "gpt-4.1-mini") -> List[Task]:
    require_openai_key()

    prompt = f"""
Estrai le ATTIVITÀ assegnate (action items) dalla trascrizione.
Regole:
- Non inventare nulla.
- Se il responsabile non è esplicito, usa "Unassigned".
- Mantieni i nomi esattamente come compaiono nel testo.
- Restituisci SOLO JSON valido (niente testo extra, niente backticks).
Formato: una lista di oggetti con chiavi:
  assignee (string), task (string), evidence (string breve citazione dal testo)

TRASCRIZIONE:
\"\"\"{transcript_normalized}\"\"\"
"""

    raw = openai_text(prompt, model=model)

    try:
        data = json.loads(raw)
    except json.JSONDecodeError:
        m = re.search(r"(\[\s*\{.*\}\s*\])", raw, flags=re.S)
        if not m:
            raise RuntimeError("LLM non ha restituito JSON valido per i task.")
        data = json.loads(m.group(1))

    tasks: List[Task] = []
    if isinstance(data, list):
        for item in data:
            if not isinstance(item, dict):
                continue
            assignee = str(item.get("assignee", "Unassigned")).strip() or "Unassigned"
            task_txt = str(item.get("task", "")).strip()
            evidence = str(item.get("evidence", "")).strip()
            if task_txt:
                tasks.append(Task(assignee=assignee, task=task_txt, evidence=evidence))

    return tasks


In [ ]:
"""POST-PROCESSING: normalizzo testo task e rimuovo duplicati"""

from typing import Tuple

def normalize_task_text(s: str) -> str:
    s = (s or "").strip()
    s = re.sub(r"\s+", " ", s)
    s = s.rstrip(".")
    return s


def dedupe_tasks(tasks: List[Task]) -> List[Task]:
    seen: set[Tuple[str, str]] = set()
    out: List[Task] = []

    for t in tasks:
        assignee = (t.assignee or "Unassigned").strip() or "Unassigned"
        task_txt = normalize_task_text(t.task)
        evidence = (t.evidence or "").strip()

        if not task_txt:
            continue

        key = (assignee.lower(), task_txt.lower())
        if key in seen:
            continue

        seen.add(key)
        out.append(Task(assignee=assignee, task=task_txt, evidence=evidence))

    return out


In [ ]:
"""OUTPUT: creo la caetella outputs/ e salvo riassunto e attività"""

import os
from dataclasses import asdict


def ensure_outdir(outdir: str) -> None:
    os.makedirs(outdir, exist_ok=True)


def save_summary_md(summary_points: List[str], out_path: str) -> None:
    lines = ["# Riassunto del Meeting", ""]
    for p in summary_points:
        lines.append(f"- {p}")
    lines.append("")
    with open(out_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))


def save_tasks_json(tasks: List[Task], out_path: str) -> None:
    payload = [asdict(t) for t in tasks]
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)


In [ ]:
"""REPORT: genero un report Markdown breve con funzionamento, risultati e limitazioni"""

def build_report_md(
    used_llm: bool,
    model_name: str,
    n_chars: int,
    n_turns: int,
    n_summary_points: int,
    n_tasks: int,
) -> str:
    mode = "LLM (OpenAI)" if used_llm else "Fallback offline (regole/euristiche)"
    return f"""# Report — Analisi della Trascrizione del Meeting

## Obiettivo
Generare un riassunto e identificare attività assegnate a partire da una trascrizione.

## Pipeline
1) Input (URL)
2) Pre-processing (normalizzazione + turni)
3) Riassunto ({mode})
4) Attività (LLM OpenAI)
5) Post-processing (dedupe)
6) Output (summary.md, tasks.json)
7) Report (report.md)

## Configurazione
- Modalità: {mode}
- Modello: {model_name if used_llm else "N/A"}
- Caratteri trascrizione: {n_chars}
- Turni stimati: {n_turns}
- Bullet riassunto: {n_summary_points}
- Task estratti: {n_tasks}

## Limitazioni
- Il pre-processing è euristico e può non separare perfettamente i turni.
- Riassunto e task dipendono dal prompt (variabilità minima).
"""


def save_report_md(report_md: str, out_path: str) -> None:
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(report_md)


In [ ]:
"""RUN end-to-end: eseguo e salvo in file outputs/"""

transcript = load_meeting_transcription(MEETING_TRANSCRIPTION_URL)
prep = preprocess_transcript(transcript)

summary_points = generate_summary(prep.transcript_normalized, model="gpt-4.1-mini")

tasks_raw = llm_extract_tasks(prep.transcript_normalized, model="gpt-4.1-mini")
tasks_final = dedupe_tasks(tasks_raw)

outdir = "outputs"
ensure_outdir(outdir)

summary_path = os.path.join(outdir, "summary.md")
tasks_path = os.path.join(outdir, "tasks.json")
report_path = os.path.join(outdir, "report.md")

save_summary_md(summary_points, summary_path)
save_tasks_json(tasks_final, tasks_path)

report_md = build_report_md(
    used_llm=has_openai_key(),
    model_name="gpt-4.1-mini",
    n_chars=len(prep.transcript_normalized),
    n_turns=len(prep.turns),
    n_summary_points=len(summary_points),
    n_tasks=len(tasks_final),
)
save_report_md(report_md, report_path)

print("Output salvati:")
print(f"- {summary_path}")
print(f"- {tasks_path}")
print(f"- {report_path}")


Output salvati:
- outputs/summary.md
- outputs/tasks.json
- outputs/report.md


In [ ]:
"""Il codice fornisce funzioni per leggere file Markdown e JSON e visualizzarne il contenuto in formato renderizzato all’interno del notebook Colab.
"""
from IPython.display import Markdown, display

def show_markdown(file_path: str):
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()
    display(Markdown(content))


def show_tasks_json(file_path: str):
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    md = ["## Attività Identificate\n"]
    for i, t in enumerate(data, 1):
        md.append(f"**{i}. {t['assignee']}**  ")
        md.append(f"- Attività: {t['task']}  ")
        if t.get("evidence"):
            md.append(f"- Evidenza: _{t['evidence']}_  ")
        md.append("")

    display(Markdown("\n".join(md)))


In [ ]:
""""Le chiamate visualizzano nel notebook il riassunto, le attività identificate e il report finale generati dall’analisi del meeting.
"""
show_markdown("outputs/summary.md")
show_tasks_json("outputs/tasks.json")
show_markdown("outputs/report.md")

# Riassunto del Meeting

- Necessità di interfaccia intuitiva per inserimento fatture.
- Gestione automatizzata scadenze con promemoria e notifiche.
- Reportistica personalizzabile con filtri e parametri selezionabili.
- Invio automatico email ai fornitori per ritardi pagamenti.
- Archiviazione digitale documenti con allegati PDF fatture.
- Uso di cloud per storage e ricerca avanzata documenti.
- Andrea redigerà piano tecnico basato sui requisiti emersi.
- Prossimo incontro per approvazione progetto definitivo.


## Attività Identificate

**1. Andrea Monti**  
- Attività: Integrare un sistema di notifiche e invio email nella piattaforma  
- Evidenza: _Andrea, possiamo integrare un sistema di notifiche e invio email nella piattaforma?_  

**2. Andrea Monti**  
- Attività: Prendere nota della necessità di uno spazio di storage e di una gestione documentale  
- Evidenza: _Andrea, puoi prendere nota di questo. Avremo bisogno di uno spazio di storage e di una gestione documentale._  

**3. Andrea Monti**  
- Attività: Utilizzare un servizio cloud come base per l’archiviazione e implementare una funzione di ricerca avanzata per i documenti  
- Evidenza: _Pensavo di utilizzare un servizio cloud come base per l’archiviazione. Potremmo implementare una funzione di ricerca avanzata per trovare rapidamente i documenti._  

**4. Andrea Monti**  
- Attività: Redigere un piano tecnico basato sui requisiti emersi  
- Evidenza: _Andrea, ti occupi di redigere un piano tecnico basato sui requisiti emersi._  


# Report — Analisi della Trascrizione del Meeting

## Obiettivo
Generare un riassunto e identificare attività assegnate a partire da una trascrizione.

## Pipeline
1) Input (URL)
2) Pre-processing (normalizzazione + turni)
3) Riassunto (LLM (OpenAI))
4) Attività (LLM OpenAI)
5) Post-processing (dedupe)
6) Output (summary.md, tasks.json)
7) Report (report.md)

## Configurazione
- Modalità: LLM (OpenAI)
- Modello: gpt-4.1-mini
- Caratteri trascrizione: 2508
- Turni stimati: 31
- Bullet riassunto: 8
- Task estratti: 4

## Limitazioni
- Il pre-processing è euristico e può non separare perfettamente i turni.
- Riassunto e task dipendono dal prompt (variabilità minima).
